# 🧪 Lasmoid 10M — Full Validation & Training (Memory-Safe for T4)

**Goal: GO / NO-GO confirmation before scaling to bigger models.**

Runs a complete diagnostic of every subsystem, trains a 10M model, and gives a verdict.
Tuned to fit a single Kaggle T4 (15GB): batch=2 + grad-accum, seq=128, streams=3,
with memory cleanup between every phase.

**Run all cells top-to-bottom. No edits required.**

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1: Memory config (MUST be before torch import) + install
# ════════════════════════════════════════════════════════════════
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle ships GPU-compatible torch — DON'T reinstall it (breaks CUDA kernels)
!pip install -q safetensors tiktoken datasets huggingface_hub

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2: Clone repo
# ════════════════════════════════════════════════════════════════
WORK_DIR = '/kaggle/working/Lasmoid'
if not os.path.exists(os.path.join(WORK_DIR, 'inference', 'model.py')):
    !git clone https://github.com/Theory903/Lasmoid.git {WORK_DIR}
else:
    print('✅ Already cloned')
os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3: Imports + logger + memory helpers
# ════════════════════════════════════════════════════════════════
import sys, json, time, traceback, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, os.path.join(WORK_DIR, 'inference'))
sys.path.insert(0, os.path.join(WORK_DIR, 'train'))
sys.path.insert(0, WORK_DIR)

RESULTS = {}

def check(name, condition, detail=''):
    icon = '✅' if condition else '❌'
    RESULTS[name] = bool(condition)
    print(f'  {icon} [{"PASS" if condition else "FAIL"}] {name}' + (f'  →  {detail}' if detail else ''))
    return condition

def section(title):
    print('\n' + '═' * 64 + f'\n  {title}\n' + '═' * 64)

def safe_run(fn, name):
    try:
        return fn()
    except Exception as e:
        RESULTS[name] = False
        print(f'  ❌ [ERROR] {name}\n     {type(e).__name__}: {e}')
        traceback.print_exc()
        return None

def free_mem():
    """Aggressively free GPU memory between phases."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def mem_gb():
    return torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

print('✅ Logger + memory helpers ready')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 1: Environment
# ════════════════════════════════════════════════════════════════
section('PHASE 1: ENVIRONMENT')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
check('PyTorch installed', True, torch.__version__)
check('GPU available', DEVICE == 'cuda',
      torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')
if DEVICE == 'cuda':
    cap = torch.cuda.get_device_capability(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    check('GPU compute >= 7.0 (torch 2.x compatible)', cap[0] >= 7,
          f'sm_{cap[0]}{cap[1]} ({total:.0f}GB)')
    if cap[0] < 7:
        print('  ⚠️  This GPU (P100/older) may be incompatible. Switch to T4 in Settings.')
check('inference/model.py exists', os.path.exists(os.path.join(WORK_DIR, 'inference', 'model.py')))
check('config_10m.json exists', os.path.exists(os.path.join(WORK_DIR, 'config_10m.json')))

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 2: Model Build
# ════════════════════════════════════════════════════════════════
section('PHASE 2: MODEL BUILD')
from dataclasses import fields as dc_fields
import transformers
from model import Lasmoid, ModelArgs, compute_loss
from optimizer import build_optimizers
from scheduler import WSDScheduler
check('Imports succeed', True)

with open(os.path.join(WORK_DIR, 'config_10m.json')) as f:
    cfg = json.load(f)
valid = {f.name for f in dc_fields(ModelArgs)}
args = ModelArgs(**{k: v for k, v in cfg.items() if k in valid})
check('Config loaded', True, f'dim={args.dim}, layers={args.n_layers}, streams={args.num_residual_streams}')

try:
    enc = transformers.PreTrainedTokenizerFast.from_pretrained(WORK_DIR)
except Exception:
    enc = transformers.PreTrainedTokenizerFast.from_pretrained(WORK_DIR, fix_mistral_regex=True)
args.vocab_size = max(args.vocab_size, len(enc))
EOS_ID = enc.eos_token_id or 1
SEQ_LEN = args.max_seq_len  # 128
check('Tokenizer loaded', True, f'vocab={len(enc)}')

torch.manual_seed(42)
model = Lasmoid(args).to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
embed_p = sum(p.numel() for n, p in model.named_parameters() if 'emb' in n.lower())
core_p = (total_p - embed_p) / 1e6
check('Model built', True, f'{total_p/1e6:.1f}M total | {core_p:.1f}M core (non-embedding)')
check('Core (non-embedding) params is ~10M', 4 <= core_p <= 16, f'{core_p:.1f}M core')
print(f'\n  Note: {embed_p/1e6:.1f}M params are the {args.vocab_size}-token embedding table.')
print(f'  GPU memory after build: {mem_gb():.2f} GB')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 3: Forward Pass (eval, no_grad — minimal memory)
# ════════════════════════════════════════════════════════════════
section('PHASE 3: FORWARD PASS')
model.eval()
B = 2
x = torch.randint(0, args.vocab_size, (B, SEQ_LEN), device=DEVICE)

def run_forward():
    with torch.no_grad():
        return model(x, x)

out = safe_run(run_forward, 'Forward pass runs')
if out is not None:
    logits, mtp_logits, concept_db, memory_state, rmaps, indices, adjs, eprobs = out
    check('Forward pass runs', True)
    check('Logits shape correct', tuple(logits.shape) == (B, SEQ_LEN, args.vocab_size), f'{tuple(logits.shape)}')
    check('Logits are finite', torch.isfinite(logits).all().item())
    check('Concept memory present', concept_db is not None,
          f'{tuple(concept_db.shape)}' if concept_db is not None else 'None')
    check('MoE routing maps present', len(rmaps) > 0, f'{len(rmaps)} layers')
    check('VQ adjacencies present', len(adjs) > 0)
    print('\n  ℹ️  MTP/CIF return None in eval mode — that is correct; they activate in train mode.')

del x, out
free_mem()

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 4: Per-Subsystem Probe
# ════════════════════════════════════════════════════════════════
section('PHASE 4: SUBSYSTEM PROBE')
torch.manual_seed(1234)
h = torch.randn(2, 16, args.dim, device=DEVICE)  # short seq for probes

def probe_norm():
    from _common import RMSNorm
    norm = RMSNorm(args.dim, args.norm_eps).to(DEVICE)
    rms = norm(h).float().square().mean(-1).sqrt().mean().item()
    return check('RMSNorm normalizes', 0.5 < rms < 2.0, f'mean RMS={rms:.3f}')
safe_run(probe_norm, 'RMSNorm')

def probe_rope():
    from _common import apply_rotary_emb
    from attention import precompute_freqs_cis
    fc = precompute_freqs_cis(args.rope_head_dim, 16, 0, args.rope_theta, 1.0, 32, 1).to(DEVICE)
    q = torch.randn(2, 16, args.n_heads, args.rope_head_dim, device=DEVICE)
    r = apply_rotary_emb(q, fc)
    return check('RoPE preserves shape & finite', r.shape == q.shape and torch.isfinite(r).all().item())
safe_run(probe_rope, 'RoPE')

def probe_ssm():
    return check('SSM branch exists', hasattr(model.layers[0], 'ssm_branch'))
safe_run(probe_ssm, 'SSM')

def probe_moe():
    from moe import Gate
    gate = Gate(0, args).to(DEVICE); gate.eval()
    flat = torch.randn(256, args.dim, device=DEVICE)
    with torch.no_grad():
        w, idx, _, _ = gate(flat)
    cov = len(set(idx.unique().tolist())) / args.n_routed_experts
    return check('MoE all experts reachable', cov >= 0.99, f'{cov*100:.0f}% coverage')
safe_run(probe_moe, 'MoE')

def probe_mhc():
    from mhc import ManifoldConstrainedHyperConnection
    n = args.num_residual_streams
    mhc = ManifoldConstrainedHyperConnection(args.dim, n, args.hc_sinkhorn_iters).to(DEVICE); mhc.eval()
    with torch.no_grad():
        _, Bmix, _ = mhc(torch.randn(2, 8, n, args.dim, device=DEVICE))
    dev = max((Bmix.sum(-1)-1).abs().max().item(), (Bmix.sum(-2)-1).abs().max().item())
    return check('mHC Sinkhorn doubly-stochastic', dev < 1e-2, f'max dev={dev:.1e}')
safe_run(probe_mhc, 'mHC')

def probe_vq():
    from vq import VectorQuantizer
    vq = VectorQuantizer(args.codebook_size, args.dim).to(DEVICE)
    q, loss, idx = vq(torch.randn(2, 8, args.dim, device=DEVICE))
    return check('VQ quantizes & computes loss', torch.isfinite(loss).item(), f'loss={loss.item():.4f}')
safe_run(probe_vq, 'VQ')

del h
free_mem()

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 5: Gradient Flow
# ════════════════════════════════════════════════════════════════
section('PHASE 5: GRADIENT FLOW')
model.train()
model.zero_grad(set_to_none=True)
x = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)
y = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)

def grad_test():
    logits, _, _, _, rmaps, _, adjs, eprobs = model(x, x)
    loss = compute_loss(logits, y, rmaps, [model.last_vq_loss], adjs, eprobs,
                        moe_aux_loss=model.last_moe_loss, ignore_index=-100)
    loss.backward()
    return loss

loss = safe_run(grad_test, 'Backward pass runs')
if loss is not None:
    check('Loss is finite', torch.isfinite(loss).item(), f'loss={loss.item():.3f}')
    total = wg = nz = nf = 0
    for n_, p in model.named_parameters():
        if not p.requires_grad:
            continue
        total += 1
        if p.grad is not None:
            wg += 1
            if not torch.isfinite(p.grad).all():
                nf += 1
            if p.grad.abs().sum() > 0:
                nz += 1
    check('Params receive gradients (>60%)', wg/total > 0.6, f'{wg/total*100:.0f}% ({wg}/{total})')
    check('No non-finite gradients', nf == 0, f'{nf} non-finite')
    print(f'  ℹ️  {nz/total*100:.0f}% have nonzero grad (MoE inactive experts + MTP are normal exceptions)')

model.zero_grad(set_to_none=True)
del x, y, loss
free_mem()

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 6: Determinism
# ════════════════════════════════════════════════════════════════
section('PHASE 6: DETERMINISM')
model.eval()
xd = torch.randint(0, args.vocab_size, (1, SEQ_LEN), device=DEVICE)

def det_test():
    torch.manual_seed(7)
    with torch.no_grad():
        o1 = model(xd, xd)[0]
    torch.manual_seed(7)
    with torch.no_grad():
        o2 = model(xd, xd)[0]
    d = (o1 - o2).abs().max().item()
    return check('Deterministic under fixed seed', d < 1e-4, f'max|Δ|={d:.2e}')
safe_run(det_test, 'Determinism')

del xd
free_mem()

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 7: Overfit Test (CRITICAL — proves learning works)
# ════════════════════════════════════════════════════════════════
section('PHASE 7: OVERFIT TEST')
print('  Memorizing a fixed batch — loss MUST drop sharply.\n')
model.train()
of_opts = build_optimizers(model, muon_lr=3e-3, adamw_lr=1e-3, weight_decay=0.0)
torch.manual_seed(0)
fx = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)
fy = torch.randint(0, args.vocab_size, (2, SEQ_LEN), device=DEVICE)
of_losses = []

def overfit():
    for i in range(60):
        for o in of_opts:
            o.zero_grad(set_to_none=True)
        logits, _, _, _, rmaps, _, adjs, eprobs = model(fx, fx)
        l = compute_loss(logits, fy, rmaps, [model.last_vq_loss], adjs, eprobs,
                         moe_aux_loss=model.last_moe_loss, ignore_index=-100)
        l.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        for o in of_opts:
            o.step()
        of_losses.append(l.item())
        if i % 15 == 0:
            print(f'    iter {i:2d} | loss {l.item():.3f}')
    return of_losses

safe_run(overfit, 'Overfit runs')
if of_losses:
    pct = (of_losses[0] - of_losses[-1]) / of_losses[0] * 100
    print(f'\n  Loss: {of_losses[0]:.3f} → {of_losses[-1]:.3f} ({pct:.0f}% drop)')
    check('Model can overfit (loss drops >40%)', pct > 40, f'{pct:.0f}% — learning confirmed')

# Clean up overfit optimizer + rebuild fresh model for real training
del of_opts, fx, fy
model.zero_grad(set_to_none=True)
del model
free_mem()
print(f'  🔄 Rebuilding fresh model... (GPU: {mem_gb():.2f}GB after cleanup)')
torch.manual_seed(42)
model = Lasmoid(args).to(DEVICE)
model.train()
print(f'  ✅ Fresh model ready ({mem_gb():.2f}GB)')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 8a: Data prep (TinyStories)
# ════════════════════════════════════════════════════════════════
section('PHASE 8: TRAINING — Data Prep')
from datasets import load_dataset

print('  📥 Downloading TinyStories...')
ds = load_dataset('roneneldan/TinyStories', split='train').select(range(25000))
toks = []
for ex in tqdm(ds, desc='tokenize'):
    t = ex.get('text', '')
    if t and len(t) > 20:
        toks.extend(enc.encode(t) + [EOS_ID])
toks = toks[:len(toks) - len(toks) % SEQ_LEN]
data = torch.tensor(toks, dtype=torch.long)
n_train = int(0.95 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
del toks, ds
gc.collect()

# Memory-safe batch settings
BATCH = 2
GRAD_ACCUM = 8   # effective batch = 16

def get_batch(split='train'):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - SEQ_LEN - 1, (BATCH,))
    xb = torch.stack([d[i:i+SEQ_LEN] for i in ix]).to(DEVICE)
    yb = torch.stack([d[i+1:i+SEQ_LEN+1] for i in ix]).to(DEVICE)
    return xb, yb, torch.ones_like(xb, dtype=torch.float32)

check('Training data ready', len(train_data) > 500_000, f'{len(train_data)//1000}K train tokens')
print(f'  Batch={BATCH} × grad_accum={GRAD_ACCUM} = {BATCH*GRAD_ACCUM} effective')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 8b: Train (gradient accumulation, memory-safe)
# ════════════════════════════════════════════════════════════════
STEPS = 3000   # optimizer steps (each = GRAD_ACCUM micro-batches)
opts = build_optimizers(model, muon_lr=3e-3, adamw_lr=5e-4, weight_decay=0.1)
sched = WSDScheduler(opts, warmup_steps=int(0.03*STEPS), stable_steps=int(0.87*STEPS),
                     decay_steps=int(0.10*STEPS),
                     base_lrs=[[g['lr'] for g in o.param_groups] for o in opts],
                     min_lr_ratio=0.1)

tokens_per_step = BATCH * GRAD_ACCUM * SEQ_LEN
print(f'\n  🚀 Training {STEPS} steps | {STEPS*tokens_per_step//1_000_000}M tokens total\n')

model.train()
losses, val_losses, val_steps = [], [], []
t0 = time.time()

for step in range(STEPS):
    sched.step(step)
    for o in opts:
        o.zero_grad(set_to_none=True)
    step_loss = 0.0
    for _ in range(GRAD_ACCUM):
        xb, yb, mask = get_batch()
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            logits, mtp, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
            l = compute_loss(logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
                             loss_mask=mask, moe_aux_loss=model.last_moe_loss, ignore_index=-100)
            if mtp is not None:
                l = l + 0.3 * F.cross_entropy(mtp.view(-1, args.vocab_size),
                                              yb[:, 1:].contiguous().view(-1), ignore_index=-100)
            l = l / GRAD_ACCUM
        l.backward()
        step_loss += l.item() * GRAD_ACCUM
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    for o in opts:
        o.step()
    losses.append(step_loss)

    if step % 300 == 0:
        model.eval()
        with torch.no_grad():
            vx, vy, vm = get_batch('val')
            vl, *_rest = model(vx, vx)
            vrm, _, vadj, vep = _rest[3], _rest[4], _rest[5], _rest[6]
            vloss = compute_loss(vl, vy, vrm, [model.last_vq_loss], vadj, vep,
                                 moe_aux_loss=model.last_moe_loss, ignore_index=-100)
        val_losses.append(vloss.item()); val_steps.append(step)
        model.train()
        if step > 0:
            avg = sum(losses[-300:]) / min(300, len(losses))
            eta = (STEPS-step) / (step/(time.time()-t0)) / 60
            print(f'  step {step:4d}/{STEPS} | train {avg:.3f} | val {vloss.item():.3f} | '
                  f'{mem_gb():.1f}GB | ETA {eta:.0f}min')

final_train = sum(losses[-100:]) / min(100, len(losses))
print(f'\n  ✅ Done in {(time.time()-t0)/60:.0f}min | train {losses[0]:.2f}→{final_train:.2f} | '
      f'val {val_losses[0]:.2f}→{val_losses[-1]:.2f}')
check('Training loss decreased >30%', (losses[0]-final_train)/losses[0] > 0.3,
      f'{(losses[0]-final_train)/losses[0]*100:.0f}% drop')
check('Val loss decreased', val_losses[-1] < val_losses[0], f'{val_losses[0]:.2f}→{val_losses[-1]:.2f}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 8c: Plot
# ════════════════════════════════════════════════════════════════
try:
    import matplotlib.pyplot as plt
    w = 30
    sm = [sum(losses[max(0,i-w):i+1])/len(losses[max(0,i-w):i+1]) for i in range(len(losses))]
    plt.figure(figsize=(10, 4))
    plt.plot(sm, label='train (smoothed)')
    plt.plot(val_steps, val_losses, 'o-', color='red', label='val', markersize=4)
    plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Lasmoid-10M'); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'(plot skipped: {e})')
free_mem()

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 9: Generation Quality
# ════════════════════════════════════════════════════════════════
section('PHASE 9: GENERATION')
from sampler import full_sample
model.eval()

@torch.no_grad()
def generate(prompt, max_tokens=50, temperature=0.7, min_p=0.05):
    tokens = enc.encode(prompt)
    generated = list(tokens)
    pad = SEQ_LEN - len(tokens)
    idx = torch.tensor([([EOS_ID]*pad)+tokens if pad > 0 else tokens[-SEQ_LEN:]],
                       dtype=torch.long, device=DEVICE)
    logits, *_ = model(idx, idx, start_pos=0)
    g = torch.Generator(device='cpu').manual_seed(42)
    for i in range(max_tokens):
        nid = full_sample(logits[:, -1, :].cpu().float(), generated,
                          temperature=temperature, min_p=min_p, generator=g)
        tid = nid.item()
        if tid == EOS_ID:
            break
        generated.append(tid)
        inp = torch.tensor([[tid]], dtype=torch.long, device=DEVICE)
        logits, *_ = model(x_enc=None, x_dec=inp, start_pos=SEQ_LEN + i)
    return enc.decode(generated[len(tokens):])

prompts = ['Once upon a time, there was a little girl named',
           'The dog ran to the park and',
           'One day, a boy found a magic']
outs = []
for p in prompts:
    o = generate(p)
    outs.append(o)
    print(f'\n  "{p}"\n   → {o[:160]}')

words = ' '.join(outs).lower().split()
uniq = len(set(words)) / max(len(words), 1)
real = sum(1 for w in words if w.isalpha() and 2 <= len(w) <= 12) / max(len(words), 1)
print('\n  Coherence:')
check('Output non-empty', len(words) > 8, f'{len(words)} words')
check('Not repetitive', 0.12 < uniq < 0.97, f'unique={uniq:.2f}')
check('>60% real-looking words', real > 0.6, f'{real*100:.0f}%')

In [ ]:
# ════════════════════════════════════════════════════════════════
# PHASE 10: VERDICT
# ════════════════════════════════════════════════════════════════
section('PHASE 10: VERDICT — GO / NO-GO')
CRITICAL = ['Forward pass runs', 'Logits shape correct', 'Logits are finite',
            'Backward pass runs', 'Loss is finite', 'No non-finite gradients',
            'Model can overfit (loss drops >40%)', 'Training loss decreased >30%',
            'MoE all experts reachable', 'mHC Sinkhorn doubly-stochastic']
QUALITY = ['Val loss decreased', 'Not repetitive', '>60% real-looking words', 'Output non-empty']

n_pass = sum(1 for v in RESULTS.values() if v)
crit_ok = all(RESULTS.get(c, False) for c in CRITICAL)
qual_ok = sum(1 for q in QUALITY if RESULTS.get(q, False))

print(f'\n  {n_pass}/{len(RESULTS)} checks passed\n')
fails = [k for k, v in RESULTS.items() if not v]
if fails:
    print('  Failed:')
    for f in fails:
        print(f'    {"🔴" if f in CRITICAL else "🟡"} {f}')
    print()

print('  ' + '─'*52)
if crit_ok and qual_ok >= 2:
    print('  🟢  VERDICT: GO — architecture validated, SAFE TO SCALE to 100M/300M')
elif crit_ok:
    print('  🟡  VERDICT: GO WITH CAUTION — all critical systems work.')
    print('      Weak generation is EXPECTED at 10M. Architecture is sound — safe to scale.')
else:
    print('  🔴  VERDICT: NO-GO — fix the 🔴 critical failures before scaling.')
print('  ' + '─'*52)

In [ ]:
# ════════════════════════════════════════════════════════════════
# Save checkpoint (+ optional HF upload)
# ════════════════════════════════════════════════════════════════
from safetensors.torch import save_file
save_dir = '/kaggle/working/lasmoid_10m'
os.makedirs(save_dir, exist_ok=True)
save_file(model.state_dict(), os.path.join(save_dir, 'model.safetensors'))
with open(os.path.join(save_dir, 'config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'💾 Saved to {save_dir}')

# Uncomment to upload to HuggingFace:
# from huggingface_hub import HfApi, login, create_repo
# try:
#     from kaggle_secrets import UserSecretsClient
#     login(token=UserSecretsClient().get_secret('HF_TOKEN'))
# except: login()
# api = HfApi(); create_repo('Theory903/lasmoid-10m-test', exist_ok=True)
# api.upload_folder(folder_path=save_dir, repo_id='Theory903/lasmoid-10m-test')
# print('☁️  Uploaded to Theory903/lasmoid-10m-test')